# Handwritten Text Generation using Character-Level RNN
### CODSOFT AI/ML Internship — Task 2
**Objective:** Build, train, and evaluate a genuine Character-Level Recurrent Neural Network (LSTM / SimpleRNN) using TensorFlow/Keras to learn sequential character probabilities and generate realistic text character by character with temperature scaling.

## 1. System Setup & Library Imports

In [ ]:
# Install dependencies (if running in a fresh Colab environment)
!pip install -q tensorflow numpy pandas matplotlib

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint

print(f"TensorFlow Version: {tf.__version__}")
print("GPU Available:", tf.config.list_physical_devices('GPU'))

## 2. Dataset Preparation & Cleaning
We provide a default literary essay on the art of handwriting. You can also upload any `.txt` or `.csv` file.

In [ ]:
raw_text = """Handwriting is an intimate mirror of the human spirit. When ink touches paper, thought becomes tangible form. Each curve, stroke, and flourish carries the rhythm of breath and contemplation. Unlike mechanical typography, cursive manuscript reveals the subtle hesitation, speed, and emotion of the writer. Throughout the centuries, scholars kept leather journals filled with ink annotations. Every letter was crafted with intention, blending discipline and creative liberty. To write by hand is to converse slowly with oneself, leaving an enduring trace upon the world. The quick brown fox jumps over the lazy dog. Continuous practice allows the neural pathways of language and fine motor control to harmonize. As words unfold line after line, stories emerge from silent thoughts."""

# Optional: Repeat to expand training volume
text = (raw_text + " ") * 4

chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: idx for idx, ch in enumerate(chars)}
idx_to_char = {idx: ch for idx, ch in enumerate(chars)}

print(f"Total Characters: {len(text):,}")
print(f"Unique Vocabulary Size: {vocab_size} characters")
print(f"Vocabulary Tokens: {chars}")

## 3. Sliding Window Sequence Creation (X -> y)

In [ ]:
seq_length = 30
step = 2
sentences = []
next_chars = []

for i in range(0, len(text) - seq_length, step):
    sentences.append(text[i: i + seq_length])
    next_chars.append(text[i + seq_length])

num_sequences = len(sentences)
print(f"Generated {num_sequences:,} training sequence pairs.")

# One-hot vectorize X and y
X = np.zeros((num_sequences, seq_length, vocab_size), dtype=np.bool_)
y = np.zeros((num_sequences, vocab_size), dtype=np.bool_)

for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_to_idx[char]] = 1
    y[i, char_to_idx[next_chars[i]]] = 1

print(f"X shape: {X.shape} (Sequences, Window Size, Vocab Size)")
print(f"y shape: {y.shape} (Sequences, Vocab Size)")

## 4. Model Construction: Character-Level LSTM

In [ ]:
model = Sequential([
    LSTM(128, input_shape=(seq_length, vocab_size)),
    Dropout(0.2),
    Dense(vocab_size, activation='softmax')
], name="Character_Level_LSTM")

model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.005),
    metrics=['accuracy']
)

model.summary()

## 5. Model Training & Loss Monitoring

In [ ]:
epochs = 35
batch_size = 64

history = model.fit(
    X, y,
    batch_size=batch_size,
    epochs=epochs,
    verbose=1
)

# Plot training loss curve
plt.figure(figsize=(10, 4))
plt.plot(range(1, epochs + 1), history.history['loss'], 'b-o', linewidth=2, label='Training Loss (Categorical Cross-Entropy)')
plt.title('Character-Level RNN: Training Loss vs Epochs', fontsize=13, fontweight='bold')
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Loss', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

## 6. Temperature-Scaled Text Generation

In [ ]:
def sample_with_temperature(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    if temperature <= 0.01:
        return np.argmax(preds)
    preds = np.log(np.maximum(preds, 1e-12)) / temperature
    exp_preds = np.exp(preds)
    probs = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, probs, 1)
    return int(np.argmax(probas))

def generate(seed_phrase, length=200, temperature=0.7):
    sentence = seed_phrase[-seq_length:].rjust(seq_length)
    generated = seed_phrase
    for _ in range(length):
        x_pred = np.zeros((1, seq_length, vocab_size))
        for t, char in enumerate(sentence):
            if char in char_to_idx:
                x_pred[0, t, char_to_idx[char]] = 1.0
        preds = model.predict(x_pred, verbose=0)[0]
        next_idx = sample_with_temperature(preds, temperature)
        next_char = idx_to_char.get(next_idx, ' ')
        generated += next_char
        sentence = sentence[1:] + next_char
    return generated

seed = "Handwriting is "
print("=" * 60)
for temp in [0.2, 0.7, 1.2]:
    print(f"\n--- Generation at Temperature T = {temp} ---")
    print(generate(seed, length=180, temperature=temp))
print("=" * 60)

## 7. Model Checkpoint Saving
Save the trained Keras model and vocabulary mapping for downstream applications.

In [ ]:
os.makedirs("saved_models", exist_ok=True)
model.save("saved_models/final_char_rnn_model.keras")

with open("saved_models/vocab.json", "w", encoding="utf-8") as f:
    json.dump({"char_to_idx": char_to_idx, "idx_to_char": {str(k): v for k, v in idx_to_char.items()}}, f, indent=2)

print("Model weights and vocabulary mappings saved to /saved_models directory.")